# 04d — EXP_04: Hybrid RAG (BGE dense + BM25 sparse fused via RRF k=60)

**Architecture:** dense + sparse retrieval fused with Reciprocal Rank Fusion → LLM. **Retriever:** `HybridRetriever` (top-5 fused chunks). **Generator:** `llama-3.3-70b-versatile` via Groq · T=0 · k=5 · max_tokens=700.

Same three-stage gating pattern as EXP_02 / EXP_03. Builds on **two empirical findings from EXP_03** ([`docs/output_notes/04c_exp03_output.md`](../docs/output_notes/04c_exp03_output.md)):

- **Complementarity**: 153 of 1,273 test questions disagree between dense (EXP_02) and sparse (EXP_03), with a 50/50 right/wrong split. Strong evidence that fusion should help.
- **Per-step pattern**: sparse beats dense on step2&3 (+1.68 pp), dense beats sparse on step1 (+1.33 pp). RRF should pick the right retriever per question.

**The EXP_01 / EXP_02 / EXP_03 baselines to compare against** (canonical test_1273):
- EXP_01 No-RAG: `Acuuracy = 0.7738`
- EXP_02 Naive Dense: `Acuuracy = 0.7573`
- EXP_03 Sparse BM25: `Acuuracy = 0.7581`

**Falsifiable hypothesis for EXP_04**: *Hybrid Acuuracy on test_1273 should land **above 0.76** (i.e., above both EXP_02 and EXP_03). If RRF correctly captures the complementarity, Hybrid should also lift Context Precision above EXP_02's 0.33.* Falsified if Hybrid Acuuracy ≤ 0.76.

---

Stages:

| Stage | Surface | Wall time | Cost | Gate |
|---|---|---|---|---|
| **A** Smoke | 50 stratified rows | ~3–5 min | $0 | always on |
| **B** Golden | 234 accepted golden rows | ~5–8 min | $0 | `RUN_GOLDEN = True` |
| **C** Test | 1,273 test-split rows (locked) | ~30–60 min | $0 | `RUN_FULL = False` until A and B clean |

Wall time is between EXP_02 (~12 min on test_1273) and EXP_03 (~97 min) — roughly Naive's BGE encode + Chroma cost + BM25's `get_scores` cost combined.

## 1. Setup

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import best_device, load_bge
from src.data.indices import load_bm25, load_chroma
from src.data.loaders import load_chunks, load_golden, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.hybrid import HybridRetriever

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)
print("Device   :", best_device())

## 2. Build the retriever (BGE-large + ChromaDB + BM25 + RRF k=60)

In [ ]:
CHROMA_DIR = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
BM25_PATH = REPO_ROOT / "data" / "indices" / "bm25.pkl"

chunks_df = load_chunks()
chroma_coll = load_chroma(CHROMA_DIR)
bm25_payload = load_bm25(BM25_PATH)
embedder = load_bge(device=best_device())
RETRIEVER = HybridRetriever(embedder, chroma_coll, bm25_payload, chunks_df)

print(f"chunks_df rows : {len(chunks_df):,}")
print(f"ChromaDB count : {chroma_coll.count():,}")
print(f"BM25 chunk_ids : {len(bm25_payload['chunk_ids']):,}")

# Sanity check on the same CAP query used by all baseline notebooks
_test_chunks = RETRIEVER.retrieve(
    "What is the first-line treatment for community-acquired pneumonia?", k=3
)
print("\nSanity query (CAP first-line treatment) — top 3:")
for c in _test_chunks:
    print(f"  RRF={c.score:.5f} {c.chunk_id} ({c.book_name}): {c.text[:90].replace(chr(10), ' ')}\u2026")

## 3. Configuration

In [ ]:
EXPERIMENT_ID = "EXP_04_HYBRID_RAG"
RESULTS_DIR = REPO_ROOT / "results"
TOP_K = 5

SMOKE_N = 50
SMOKE_SEED = 42  # same seed as EXP_01/02/03 → directly comparable smoke surfaces

RUN_SMOKE = True
RUN_GOLDEN = True
RUN_FULL = False  # Stage C · ~30–60 min Groq on test split (1,273)

## 4. Stage A — Smoke (50 stratified rows)

In [ ]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)

    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    smoke_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=smoke,
        output_dir=RESULTS_DIR / "exp_04_hybrid_rag__smoke_50",
        experiment_id=EXPERIMENT_ID,
        dataset_label="smoke_50",
        k=TOP_K,
    )
    print(json.dumps(smoke_summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

**Acceptance gate (Stage A):** `n_questions == 50`; `Acuuracy ≥ 0.75` (smoke 50 should be in the same ballpark as EXP_02/03 smoke); 0 null `pred_letter`s; `retrieval.jsonl` rows have 5 chunk IDs each with non-zero RRF scores (typically in [0.005, 0.035] given RRF k=60).

## 5. Stage B — Golden 234

In [ ]:
if RUN_GOLDEN:
    golden = load_golden()
    print(f"Golden surface: {len(golden)} accepted rows")

    golden_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=golden,
        output_dir=RESULTS_DIR / "exp_04_hybrid_rag__golden_234",
        experiment_id=EXPERIMENT_ID,
        dataset_label="golden_234",
        k=TOP_K,
    )
    print(json.dumps(golden_summary, indent=2))
else:
    print("RUN_GOLDEN = False — skipping Stage B")

## 6. Stage C — Test split (1,273 rows · gated · ~30–60 min)

Flip `RUN_FULL = True` only after Stages A and B look right.

In [ ]:
if RUN_FULL:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    full_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_04_hybrid_rag__test_1273",
        experiment_id=EXPERIMENT_ID,
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(full_summary, indent=2))
else:
    print("RUN_FULL = False — skipping Stage C")

## 7. Inspect — does Hybrid clear the falsifiable bar?

In [ ]:
import pandas as pd

for label in ("smoke_50", "golden_234", "test_1273"):
    p = RESULTS_DIR / f"exp_04_hybrid_rag__{label}" / "predictions.jsonl"
    if not p.exists():
        continue
    preds = pd.DataFrame(json.loads(line) for line in p.read_text().splitlines())
    print(f"\n=== {label}  (n={len(preds)}) ===")
    print(f"  EXP_04 Hybrid Acuuracy : {preds['is_correct'].mean():.4f}")
    print(f"  mean latency           : {preds['latency_s'].mean():.3f} s")

    if label == "test_1273":
        # Compare against EXP_01/02/03 reference numbers
        e1 = json.loads((REPO_ROOT / 'results' / 'exp_01_base_llm__test_1273' / 'summary.json').read_text())
        e2 = json.loads((REPO_ROOT / 'results' / 'exp_02_naive_rag__test_1273' / 'summary.json').read_text())
        e3 = json.loads((REPO_ROOT / 'results' / 'exp_03_sparse_rag__test_1273' / 'summary.json').read_text())
        e4_acc = preds['is_correct'].mean()
        print()
        print(f'  Hypothesis check — Hybrid > 0.76?  {"\u2713 SUPPORTED" if e4_acc > 0.76 else "\u2717 FALSIFIED"}')
        print(f'    EXP_01 No-RAG       : {e1["Acuuracy"]:.4f}')
        print(f'    EXP_02 Naive Dense  : {e2["Acuuracy"]:.4f}')
        print(f'    EXP_03 Sparse BM25  : {e3["Acuuracy"]:.4f}')
        print(f'    EXP_04 Hybrid       : {e4_acc:.4f}')

        medqa = load_medqa_4opt().reset_index(drop=True)
        joined = preds.merge(medqa[["question_id", "meta_info"]], on="question_id", how="left")
        print("\n  by meta_info (vs other archs):")
        for mi, g in joined.groupby("meta_info"):
            print(f'    {mi:10s}: EXP_04={g["is_correct"].mean():.4f}  (n={len(g)})')

---

**Next.** Run [`04d_exp04_ragas.ipynb`](04d_exp04_ragas.ipynb) when EXP_04 + EXP_05 baselines are both done — user is batching all 4 RAG architectures' RAGAS evaluations.